In [28]:
import random
import pandas as pd
import numpy as np
from collections import defaultdict

In [29]:
# Load dataset
df = pd.read_csv('data/soc-sign-bitcoinotc.csv.gz', names=["SOURCE", "TARGET", "RATING", "TIME"])
df['TIME'] = pd.to_datetime(df['TIME'])

# Create neighbor mappings
outgoing_neighbors = df.groupby('SOURCE')['TARGET'].apply(set).to_dict()
incoming_neighbors = df.groupby('TARGET')['SOURCE'].apply(set).to_dict()

# Ensure all nodes are included, even if they are only targets
all_nodes = set(outgoing_neighbors.keys()).union(set(incoming_neighbors.keys()))

# Add missing nodes to outgoing_neighbors with no outgoing edges
for node in all_nodes:
    if node not in outgoing_neighbors:
        outgoing_neighbors[node] = set()

In [30]:
# Initialize the DLAS algorithm
def dlas(graph, num_iterations, sample_ratio):
    """
    DLAS algorithm for graph sampling.

    Parameters:
        graph (dict): The outgoing neighbors mapping (adjacency list).
        num_iterations (int): Number of iterations for the algorithm.
        sample_ratio (float): Percentage of nodes to sample from the graph.

    Returns:
        set: Sampled nodes.
    """
    nodes = list(graph.keys())
    num_nodes = len(nodes)
    sample_size = int(num_nodes * sample_ratio)
    
    # Initialize probability vectors for all nodes
    probabilities = {node: 1 / num_nodes for node in nodes}

    # Start from a randomly chosen node
    sampled_nodes = set()
    current_node = random.choice(nodes)

    for _ in range(num_iterations):
        # If we've sampled enough nodes, break the loop
        if len(sampled_nodes) >= sample_size:
            break

        # Choose the next action (neighbor node) based on probabilities
        neighbors = graph.get(current_node, [])
        
        # Exclude already sampled nodes from neighbors
        neighbors = [n for n in neighbors if n not in sampled_nodes]

        if not neighbors:
            # If no unvisited neighbors, pick a random unvisited node
            unvisited_nodes = [node for node in nodes if node not in sampled_nodes]
            if not unvisited_nodes:
                break  # All nodes have been sampled
            current_node = random.choice(unvisited_nodes)
            continue

        # Normalize probabilities for the neighbors
        probabilities_sum = sum(probabilities[n] for n in neighbors)
        normalized_probs = [probabilities[n] / probabilities_sum for n in neighbors]

        # Select the next node based on probabilities
        selected_node = random.choices(neighbors, weights=normalized_probs)[0]

        # Add the selected node to the sampled set
        sampled_nodes.add(selected_node)

        # Update the probabilities
        probabilities[selected_node] = 0  # Set probability to zero to avoid revisiting

        # Optionally, redistribute the probability mass to unvisited nodes
        total_prob = sum(probabilities.values())
        probabilities = {node: prob / total_prob for node, prob in probabilities.items()}

        # Move to the selected node
        current_node = selected_node

    return sampled_nodes

In [31]:
# Apply the DLAS algorithm with increased iterations
sampled_nodes = dlas(outgoing_neighbors, num_iterations=10000, sample_ratio=0.3)

# Output the sampled nodes
print(f"Sampled Nodes ({len(sampled_nodes)}): {sampled_nodes}")


Sampled Nodes (1764): {1, 2, 3, 4, 6, 7, 13, 17, 19, 21, 23, 25, 26, 28, 29, 33, 35, 36, 37, 39, 41, 45, 51, 54, 57, 60, 61, 62, 64, 68, 69, 75, 76, 77, 78, 80, 81, 87, 93, 95, 96, 97, 99, 100, 104, 107, 109, 110, 111, 112, 113, 114, 115, 120, 125, 129, 132, 133, 135, 141, 142, 143, 144, 146, 147, 149, 152, 156, 159, 162, 163, 164, 166, 171, 174, 178, 179, 180, 181, 183, 184, 185, 186, 192, 198, 199, 200, 202, 203, 204, 206, 209, 214, 215, 216, 219, 221, 222, 223, 224, 225, 227, 228, 230, 233, 235, 242, 243, 245, 249, 250, 251, 256, 257, 261, 262, 263, 266, 268, 269, 270, 272, 273, 274, 275, 278, 279, 280, 283, 288, 293, 295, 296, 298, 300, 301, 304, 307, 308, 309, 312, 313, 318, 319, 320, 324, 325, 331, 340, 346, 348, 350, 353, 359, 361, 363, 364, 366, 372, 374, 383, 384, 386, 387, 390, 391, 393, 394, 395, 396, 399, 401, 402, 403, 404, 405, 408, 411, 412, 415, 416, 417, 418, 421, 423, 425, 427, 428, 430, 432, 433, 436, 437, 443, 445, 450, 452, 453, 454, 455, 462, 463, 465, 467, 468, 4